In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score, accuracy_score)
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

In [ ]:
# Load dataset
df = pd.read_csv("data.csv")
print("Original shape:", df.shape)
print("\nClass distribution:\n", df['is_fraud'].value_counts())

# Sample to make it manageable (keep all fraud + 200k legit)
fraud = df[df['is_fraud'] == 1]
legit = df[df['is_fraud'] == 0].sample(n=200000, random_state=42)
df = pd.concat([fraud, legit]).sample(frac=1, random_state=42).reset_index(drop=True)
print("\nSampled shape:", df.shape)
print("\nSampled class distribution:\n", df['is_fraud'].value_counts())

In [ ]:
# Drop irrelevant columns
drop_cols = ['ssn', 'cc_num', 'first', 'last', 'street', 'city', 'zip',
             'dob', 'acct_num', 'profile', 'trans_num', 'trans_date',
             'trans_time', 'unix_time', 'merchant', 'job']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Encode categorical columns
cat_cols = ['gender', 'category', 'state']
df = pd.get_dummies(df, columns=[c for c in cat_cols if c in df.columns], drop_first=True)

print("Shape after preprocessing:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
# Visualize class distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='is_fraud', data=df, palette='Set2')
plt.title('Class Distribution (0 = Legit, 1 = Fraud)')
plt.xlabel('Is Fraud')
plt.ylabel('Count')
plt.xticks([0, 1], ['Legit', 'Fraud'])
plt.tight_layout()
plt.show()

print(f"\nLegit transactions: {len(df[df['is_fraud']==0]):,}")
print(f"Fraud transactions: {len(df[df['is_fraud']==1]):,}")
print(f"Fraud percentage: {df['is_fraud'].mean()*100:.2f}%")

In [ ]:
# Split features and target
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

# Train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("\nTraining class distribution:\n", y_train.value_counts())
print("\nTest class distribution:\n", y_test.value_counts())

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE to balance training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", pd.Series(y_train_res).value_counts().to_dict())
print("\nTraining set size after SMOTE:", X_train_res.shape)

In [ ]:
# Train Logistic Regression
print("Training Logistic Regression)")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_res, y_train_res)

# Predict
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("\n=== Logistic Regression Results ===")
print(classification_report(y_test, y_pred_lr))
print("AUC-ROC Score:", round(roc_auc_score(y_test, y_prob_lr), 4))

In [ ]:
# Train Random Forest
print("Training Random Forest")
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', 
                             random_state=42, n_jobs=-1)
rf.fit(X_train_res, y_train_res)

# Predict
y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

print("\n=== Random Forest Results ===")
print(classification_report(y_test, y_pred_rf))
print("AUC-ROC Score:", round(roc_auc_score(y_test, y_prob_rf), 4))

In [ ]:
# ROC Curve Comparison
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, y_prob_lr):.4f})', color='blue')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, y_prob_rf):.4f})', color='green')
plt.plot([0,1],[0,1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression vs Random Forest')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Final Results Summary
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [accuracy_score(y_test, y_pred_lr), accuracy_score(y_test, y_pred_rf)],
    'Precision': [precision_score(y_test, y_pred_lr), precision_score(y_test, y_pred_rf)],
    'Recall': [recall_score(y_test, y_pred_lr), recall_score(y_test, y_pred_rf)],
    'F1 Score': [f1_score(y_test, y_pred_lr), f1_score(y_test, y_pred_rf)],
    'AUC-ROC': [roc_auc_score(y_test, y_prob_lr), roc_auc_score(y_test, y_prob_rf)]
})

# Round to 4 decimal places
results.iloc[:, 1:] = results.iloc[:, 1:].round(4)
print(results.to_string(index=False))